# Track B agent worker on a Kaggle GPU

Runs the agent worker (Whisper STT → intake/LLM → Piper TTS → MuseTalk lip-synced face) on this notebook and connects
**outbound** to your LiveKit Cloud project. Kaggle has no inbound ports, so nothing here listens:
the browser and the API on your PC create the room, LiveKit dispatches the job to this worker.

**Before running**
1. Notebook settings → **Accelerator: GPU T4 x2**, **Internet: On** (needs a phone-verified account).
2. Add-ons → **Secrets**, attach: `TRACKB_GEMINI_API_KEY`, `TRACKB_LIVEKIT_URL`,
   `TRACKB_LIVEKIT_API_KEY`, `TRACKB_LIVEKIT_API_SECRET`.
3. **Stop the worker on your PC** — both register as `trackb-intake` in the same project, and
   LiveKit would send each session to whichever it picks.

**When done:** run the last cell (or stop the session) — an idle notebook still burns GPU quota.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
REPO = "/kaggle/working/Rooman-Agent"
!rm -rf {REPO} && git clone -q --depth 1 -b feature/realtime-voice-loop https://github.com/daneshrooman-del/Rooman-Agent.git {REPO}
!cd {REPO} && git log --oneline -1

In [ ]:
# Runtime deps (sentence-transformers is skipped: only knowledge-base ingestion imports it, lazily).
# nvidia-cublas/cudnn: the CUDA libraries ctranslate2 (faster-whisper) loads on the GPU.
!pip install -q "faster-whisper>=1.0" "piper-tts>=1.8" "livekit-agents>=1.8" livekit-api \
    google-genai redis langgraph qdrant-client sqlmodel structlog tenacity pypdf \
    python-multipart pydantic-settings httpx "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"
!pip install -q --no-deps -e {REPO}

In [ ]:
# Redis holds resumable session state (trackb.session.redis_store).
!apt-get -qq install -y redis-server > /dev/null 2>&1; redis-server --daemonize yes > /dev/null; sleep 1; redis-cli ping

In [ ]:
import os, subprocess, urllib.request
MODELS = "/kaggle/working/models/piper"
os.makedirs(MODELS, exist_ok=True)
base = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/"
for name in ["en_US-lessac-medium.onnx", "en_US-lessac-medium.onnx.json"]:
    path = f"{MODELS}/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(base + name, path)
print(os.listdir(MODELS))

## The agent's face (MuseTalk lip-sync)

`AVATAR` picks the face: `"musetalk"` (real lip-sync on both T4s — 28.8 fps with the TAESD decoder,
measured in `musetalk_benchmark.ipynb`), `"placeholder"` (simple CPU face, no setup) or `"none"`
(voice only). The MuseTalk cells below install it, download its weights (~4 GB) and prepare the
sample avatar once (~2–3 min); they do nothing for the other two options.

Only the mouth is generated. Everything else (eyes, head, hands) is the reference video playing in
a loop -- so the choice of `AVATAR_VIDEO` decides how alive the avatar looks.

In [ ]:
AVATAR = "musetalk"   # "musetalk" | "placeholder" | "none"
# Reference video: MuseTalk regenerates only the mouth; blinks, head motion and hand gestures all
# come from this video, looped. For gestures use a 30-60 s waist-up clip of the person talking with
# their hands, face to camera. Upload it via Add Input -> Upload and put its /kaggle/input/... path.
AVATAR_VIDEO = "data/video/sun.mp4"   # path relative to MuseTalk, or absolute
AVATAR_ID = "sun"                     # change when you change the video (new preparation)
FRAMING = "full"                      # "full" = whole frame (body/hands) | "head" = face crop
AVATAR_FPS = "20"                     # 2xT4 + TAESD measured 28.8 fps; 20 leaves headroom live
# Or start from a single photo: set PHOTO to its /kaggle/input/... path (upload via Add Input ->
# Upload). LivePortrait animates it once into a short clip with blinks and small head movement,
# which then replaces AVATAR_VIDEO. Front-facing, sharp, mouth ideally closed. No hand gestures
# are possible from a photo. Only use photos you have the rights/consent to animate.
PHOTO = ""                            # e.g. "/kaggle/input/my-avatar/photo.png"
MT = "/kaggle/working/MuseTalk"
MT_AVATAR = f"{MT}/results/v15/avatars/{AVATAR_ID}"

In [ ]:
if AVATAR == "musetalk":
    import subprocess
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode:
            print(r.stdout[-1500:], r.stderr[-1500:])
        return r.returncode
    sh(f"rm -rf {MT} && git clone -q --depth 1 https://github.com/TMElyralab/MuseTalk.git {MT}")
    # MuseTalk's requirements.txt pins versions that don't exist for Python 3.13 (numpy 1.23,
    # tensorflow 2.12); install the libraries it actually imports, unpinned, on top of Kaggle's torch.
    sh("pip install -q diffusers accelerate transformers einops omegaconf librosa soundfile "
       "opencv-python-headless 'imageio[ffmpeg]' ffmpeg-python huggingface_hub gdown face-alignment")
    print("MuseTalk:", sh(f"cd {MT} && git log --oneline -1") == 0)

In [ ]:
if AVATAR == "musetalk":
    # Replace MuseTalk's mmpose-based landmark step with our face-alignment version (changelog 014).
    import urllib.request
    PATCH = ("https://raw.githubusercontent.com/daneshrooman-del/Rooman-Agent/"
             "feature/realtime-voice-loop/kaggle/musetalk_patches/preprocessing.py")
    urllib.request.urlretrieve(PATCH, f"{MT}/musetalk/utils/preprocessing.py")
    print("patched:", "face_alignment" in open(f"{MT}/musetalk/utils/preprocessing.py").read())

In [ ]:
if AVATAR == "musetalk":
    # Weights, laid out the way MuseTalk's download_weights.sh expects (models/...).
    import os, urllib.request
    from huggingface_hub import hf_hub_download
    M = f"{MT}/models"
    def hf(repo, filename, dest_dir, **kw):
        os.makedirs(dest_dir, exist_ok=True)
        return hf_hub_download(repo, filename, local_dir=dest_dir, **kw)

    for f in ["musetalkV15/unet.pth", "musetalkV15/musetalk.json"]:
        hf("TMElyralab/MuseTalk", f, M)
    for f in ["config.json", "diffusion_pytorch_model.bin"]:
        hf("stabilityai/sd-vae-ft-mse", f, f"{M}/sd-vae")
    for f in ["config.json", "pytorch_model.bin", "preprocessor_config.json"]:
        hf("openai/whisper-tiny", f, f"{M}/whisper")
    os.makedirs(f"{M}/face-parse-bisent", exist_ok=True)
    get_ipython().system(f"gdown -q --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O {M}/face-parse-bisent/79999_iter.pth")
    urllib.request.urlretrieve("https://download.pytorch.org/models/resnet18-5c106cde.pth",
                               f"{M}/face-parse-bisent/resnet18-5c106cde.pth")
    get_ipython().system(f"find {M} -type f | sed \"s|{M}/||\" | sort")

In [ ]:
if AVATAR == "musetalk" and PHOTO:
    # Photo -> ~12 s "alive" reference clip with LivePortrait (one-time; seconds on a T4).
    # Driving clip d13 = calm face with small natural head movement (most bundled clips are
    # exaggerated expressions). --flag_normalize_lip closes the lips first, so a toothy smile in
    # the photo doesn't carry into the talking mouth; --driving_multiplier 0.8 tones motion down.
    import glob, os, shutil, subprocess
    if not os.path.isfile(PHOTO):
        found = [f for f in glob.glob("/kaggle/input/**/*", recursive=True)
                 if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))]
        raise FileNotFoundError(
            f"PHOTO must be a file path on disk, not a web address: {PHOTO!r} not found. "
            "Attach the dataset (Add Input), then use one of: " + (", ".join(found[:10]) or
            "(no images under /kaggle/input -- is the dataset attached?)"))
    # A worker left running from an earlier run holds MuseTalk in RAM and on both GPUs; stop it
    # so LivePortrait has room (running out of memory ends the whole Kaggle session).
    if "worker" in globals() and worker.poll() is None:
        worker.kill(); worker.wait(); print("stopped the running worker")
    LP = "/kaggle/working/LivePortrait"
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode:
            print(r.stdout[-2000:], r.stderr[-2000:])
        return r.returncode
    if not os.path.exists(f"{LP}/pretrained_weights/liveportrait"):
        sh(f"rm -rf {LP} && git clone -q --depth 1 https://github.com/KwaiVGI/LivePortrait.git {LP}")
        # Its requirements pin versions that don't exist for Python 3.13; these are what it imports
        # (checked by running it on CPU locally, changelog 021).
        # CPU onnxruntime (already installed for Piper/faster-whisper) is enough: it only runs face
        # detection. Installing onnxruntime-gpu next to it is a known source of crashes.
        sh("pip install -q tyro pykalman rich lmdb albumentations onnx "
           "imageio-ffmpeg ffmpeg-python scikit-image requests")
        from huggingface_hub import snapshot_download
        snapshot_download("KwaiVGI/LivePortrait", local_dir=f"{LP}/pretrained_weights",
                          ignore_patterns=["*.git*", "README.md", "docs/*", "*animal*",
                                           "liveportrait_animals/*"])
    name = os.path.splitext(os.path.basename(PHOTO))[0]
    out_dir = "/kaggle/working/photo_avatar"
    rc = sh(f"cd {LP} && python inference.py -s '{PHOTO}' -d assets/examples/driving/d13.mp4 "
            f"-o {out_dir} --flag_normalize_lip --driving_multiplier 0.8")
    clip = f"{out_dir}/{name}--d13.mp4"
    assert rc == 0 and os.path.exists(clip), "LivePortrait failed -- see output above"
    AVATAR_VIDEO = clip
    AVATAR_ID = f"photo-{name}"
    MT_AVATAR = f"{MT}/results/v15/avatars/{AVATAR_ID}"
    print("animated photo ->", AVATAR_VIDEO, "| avatar id", AVATAR_ID)

In [ ]:
if AVATAR == "musetalk":
    # One-time avatar preparation (face boxes, latents, blend masks) with MuseTalk's own script.
    # audio_clips is empty, so it prepares and exits without rendering anything.
    import os
    if not os.path.exists(f"{MT_AVATAR}/latents.pt"):
        open(f"{MT}/configs/inference/prepare_only.yaml", "w").write(
            f'{AVATAR_ID}:\n  preparation: True\n  bbox_shift: 5\n  video_path: "{AVATAR_VIDEO}"\n  audio_clips: {{}}\n')
        r = subprocess.run(
            f"cd {MT} && TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python -m scripts.realtime_inference "
            "--inference_config configs/inference/prepare_only.yaml --result_dir results/realtime "
            "--unet_model_path models/musetalkV15/unet.pth --unet_config models/musetalkV15/musetalk.json "
            "--version v15 --fps 25", shell=True, capture_output=True, text=True)
        print("exit", r.returncode, (r.stdout + r.stderr)[-1500:] if r.returncode else "")
    print("prepared avatar:", sorted(os.listdir(MT_AVATAR)) if os.path.exists(MT_AVATAR) else "MISSING")

In [ ]:
# Secrets -> environment for the worker process. Values are never printed.
from kaggle_secrets import UserSecretsClient
import importlib

def _lib_dir(module_name):
    """Directory of a pip-installed NVIDIA lib, or None. These are namespace packages, so
    `__file__` is None -- use `__path__`."""
    try:
        mod = importlib.import_module(module_name)
    except ImportError:
        return None
    paths = list(getattr(mod, "__path__", []))
    return paths[0] if paths else None

secrets = UserSecretsClient()
env = dict(os.environ)
for name in ["TRACKB_GEMINI_API_KEY", "TRACKB_LIVEKIT_URL",
             "TRACKB_LIVEKIT_API_KEY", "TRACKB_LIVEKIT_API_SECRET"]:
    env[name] = secrets.get_secret(name)

cuda_libs = [d for d in (_lib_dir("nvidia.cublas.lib"), _lib_dir("nvidia.cudnn.lib")) if d]
# Avatar settings fall back to defaults if a line went missing from the avatar cell.
env.update({
    "TRACKB_LLM_PROVIDER": "gemini",
    "TRACKB_GEMINI_MODEL": "gemini-flash-lite-latest",
    "TRACKB_WHISPER_DEVICE": "cuda",
    "TRACKB_WHISPER_COMPUTE_TYPE": "float16",
    "TRACKB_WHISPER_MODEL_SIZE": "small.en",   # GPU can afford better accuracy than CPU tiny.en
    "TRACKB_WHISPER_LANGUAGE": "en",
    "TRACKB_TTS_VOICE_MODEL_PATH": f"{MODELS}/en_US-lessac-medium.onnx",
    "TRACKB_REDIS_URL": "redis://localhost:6379/0",
    "TRACKB_AVATAR_RENDERER": AVATAR,
    "TRACKB_MUSETALK_DIR": MT,
    "TRACKB_MUSETALK_AVATAR_DIR": MT_AVATAR,
    "TRACKB_MUSETALK_DEVICES": '["cuda:0", "cuda:1"]',
    "TRACKB_MUSETALK_DECODER": "taesd",
    "TRACKB_AVATAR_FRAMING": globals().get("FRAMING", "full"),
    "TRACKB_AVATAR_FPS": str(globals().get("AVATAR_FPS", "20")),
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",   # MuseTalk's legacy-format checkpoints (changelog 014)
    "TRACKB_WORKER_INIT_TIMEOUT_SECONDS": "300",  # MuseTalk on two GPUs loads in ~45 s
    "LD_LIBRARY_PATH": ":".join(cuda_libs + [env.get("LD_LIBRARY_PATH", "")]),
})
CONFIG_KEYS = ["TRACKB_LLM_PROVIDER", "TRACKB_GEMINI_MODEL", "TRACKB_WHISPER_DEVICE", "TRACKB_AVATAR_RENDERER",
               "TRACKB_WHISPER_MODEL_SIZE", "TRACKB_TTS_VOICE_MODEL_PATH", "TRACKB_LIVEKIT_URL"]
for k in CONFIG_KEYS:
    print(f"{k:30} {env[k]}")
print("secrets loaded:", all(env.get(n) for n in ["TRACKB_GEMINI_API_KEY", "TRACKB_LIVEKIT_API_KEY",
                                                  "TRACKB_LIVEKIT_API_SECRET"]))

In [ ]:
# GPU sanity check in a separate process (same env the worker gets): load Whisper on CUDA and
# time a transcription. If this fails with a cuDNN/cuBLAS error, set TRACKB_WHISPER_DEVICE=cpu
# and TRACKB_WHISPER_COMPUTE_TYPE=int8 above -- Kaggle's CPU is fast enough to keep going.
check = '''
import time, numpy as np
from faster_whisper import WhisperModel
t = time.time(); m = WhisperModel("small.en", device="cuda", compute_type="float16")
print(f"loaded in {time.time()-t:.1f}s")
audio = (np.random.randn(16000 * 5) * 0.01).astype(np.float32)
t = time.time(); list(m.transcribe(audio, language="en")[0]); print(f"5 s of audio in {time.time()-t:.2f}s")
'''
print(subprocess.run(["python", "-c", check], env=env, capture_output=True, text=True, timeout=600).stdout)

In [ ]:
# Start the agent worker (production mode). It connects out to LiveKit Cloud and waits for jobs.
# Refuses to start on a half-configured env (e.g. if the cell above failed part-way), and stops a
# worker left over from an earlier run of this cell.
missing = [k for k in CONFIG_KEYS if not env.get(k)]
assert not missing, f"config cell didn't complete, missing: {missing}"
# A restarted Kaggle session wipes installed packages, Redis and downloaded models; re-running only
# the later cells then starts a worker that dies on its first import. Check the essentials first.
# Checked in a fresh Python with the worker's env, as the worker will run: this kernel started
# before `pip install -e` and doesn't see the editable `trackb` install, a new process does.
need = ["structlog", "livekit.agents", "faster_whisper", "piper", "trackb"]
if AVATAR == "musetalk":
    need += ["diffusers", "cv2", "face_alignment"]
probe = ("import importlib.util, sys; "
         "print(','.join(m for m in sys.argv[1:] if importlib.util.find_spec(m) is None))")
absent = [m for m in subprocess.run(["python", "-c", probe, *need], env=env, cwd=REPO,
                                    capture_output=True, text=True).stdout.strip().split(",") if m]
assert not absent, (f"missing packages {absent}: the Kaggle session was reset -- "
                    "run all cells from the top (Run All)")
assert subprocess.run("redis-cli ping", shell=True, capture_output=True).returncode == 0, \
    "Redis isn't running: the session was reset -- run all cells from the top (Run All)"
# A restarted Kaggle session wipes installed packages, Redis and downloaded models; re-running only
# the later cells then starts a worker that dies on its first import. Check the essentials first.
# Checked in a fresh Python with the worker's env, as the worker will run: this kernel started
# before `pip install -e` and doesn't see the editable `trackb` install, a new process does.
need = ["structlog", "livekit.agents", "faster_whisper", "piper", "trackb"]
if AVATAR == "musetalk":
    need += ["diffusers", "cv2", "face_alignment"]
probe = ("import importlib.util, sys; "
         "print(','.join(m for m in sys.argv[1:] if importlib.util.find_spec(m) is None))")
absent = [m for m in subprocess.run(["python", "-c", probe, *need], env=env, cwd=REPO,
                                    capture_output=True, text=True).stdout.strip().split(",") if m]
assert not absent, (f"missing packages {absent}: the Kaggle session was reset -- "
                    "run all cells from the top (Run All)")
assert subprocess.run("redis-cli ping", shell=True, capture_output=True).returncode == 0, \
    "Redis isn't running: the session was reset -- run all cells from the top (Run All)"
if "worker" in globals() and worker.poll() is None:
    worker.terminate()
    try:
        worker.wait(timeout=10)
    except subprocess.TimeoutExpired:
        worker.kill(); worker.wait()
    print("stopped previous worker")

LOG = "/kaggle/working/worker.log"
worker = subprocess.Popen(["python", "-m", "trackb.session.entrypoint", "start"],
                          env=env, cwd=REPO, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
import time; time.sleep(90 if AVATAR == "musetalk" else 30)  # MuseTalk prewarm ~45-60 s
print("worker pid", worker.pid, "running" if worker.poll() is None else f"EXITED {worker.returncode}")
!grep -E "models_prewarmed|models_prewarm_failed|registered worker|ERROR|Traceback" {LOG} | cut -c1-200

Now open the test page on your PC (`http://localhost:8765/voice-test.html`, API running locally
against the same LiveKit Cloud project) and start an intake session. Re-run the cell below to
watch this worker handle it.

In [ ]:
!grep -vE "DEBUG" {LOG} | grep -E "stt_endpoint|stt_transcript|text_message|intake_|tts_synth|session_|ERROR|Error|Traceback" | tail -30 | cut -c1-300

In [ ]:
# Stop the worker (and stop the Kaggle session afterwards to save GPU hours).
# On SIGTERM LiveKit drains: it waits for live sessions to end. Give it a few seconds, then kill.
worker.terminate()
try:
    worker.wait(timeout=10)
except subprocess.TimeoutExpired:
    worker.kill(); worker.wait()
print("stopped", worker.returncode)